In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processor loaded successfully!


Loading weights: 100%|██████████| 686/686 [00:02<00:00, 240.60it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                        


Model loaded successfully!


In [2]:
# ============ OPENCHAIR ============
CACHE_DIR = None

OUTPUT_JSON = "../results/infer/openchair/llava15/normal.json"
OUTPUT_CSV = "../results/infer/openchair/llava15/normal.csv"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 256
PROMPT = "Describe this image."

In [4]:
import json
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import load_dataset

processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def clean_output(text):
    text = text.strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()


def save_json(rows, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

och_dataset = load_dataset(
    "moranyanuka/OpenCHAIR",
    cache_dir=CACHE_DIR
)["test"]

print(och_dataset)
print("Columns:", och_dataset.column_names)
print("Num samples:", len(och_dataset))

model.eval()

hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

indices = list(range(len(och_dataset)))
results = []

for batch_indices in tqdm(list(batch_list(indices, BATCH_SIZE))):
    images = []
    prompts = []
    valid_indices = []

    for idx in batch_indices:
        image = och_dataset[idx]["image"]

        if isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            image = Image.open(image).convert("RGB")

        images.append(image)
        prompts.append(hf_prompt)
        valid_indices.append(idx)

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    device = next(model.parameters()).device
    inputs = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    for idx, output in zip(valid_indices, outputs):
        caption = clean_output(output)

        results.append({
            "index": idx,
            "generated_caption": caption,
        })


    # save after each batch, useful in notebook in case runtime stops
    results = sorted(results, key=lambda x: x["index"])
    save_json(results, OUTPUT_JSON)

print(f"Saved {len(results)} captions to {OUTPUT_JSON}")

# ============ CONVERT JSON TO OPENCHAIR CSV ============

with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    rows = json.load(f)

rows = sorted(rows, key=lambda x: x["index"])

df = pd.DataFrame({
    "generated_caption": [row["generated_caption"] for row in rows]
})

Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved CSV to {OUTPUT_CSV}")
df.head()

Generating test split: 100%|██████████| 4863/4863 [00:01<00:00, 2729.87 examples/s]


Dataset({
    features: ['image', 'text'],
    num_rows: 4863
})
Columns: ['image', 'text']
Num samples: 4863


100%|██████████| 304/304 [53:20<00:00, 10.53s/it]

Saved 4863 captions to ../results/infer/openchair/llava15/normal.json
Saved CSV to ../results/infer/openchair/llava15/normal.csv


,generated_caption
0,The image depicts a dogfight between two fight...
1,The image features a young boy riding a bicycl...
2,"The image features a group of people, both men..."
3,"The image features a woman with red hair, wear..."
4,The image features a woman standing on a sidew...
